# Urban SDK — Traffic Speed Visualization

Visualizes traffic speed data for Duval County, FL (Jan 1, 2024) using the Urban SDK Geospatial API and MapboxGL.

**Prerequisites:**
- `docker-compose up` is running in `../app/`
- Ingestion script has been executed: `docker-compose exec api python -m src.scripts.ingest`
- Fill in your Mapbox token below

In [ ]:
# Cell 1: Imports and configuration
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from mapboxgl.utils import create_color_stops
from mapboxgl.viz import LinestringViz

MAPBOX_TOKEN = "your_mapbox_token_here"  # Replace with your token from https://mapbox.com
BASE_URL = "http://localhost:8000"

# Verify API is reachable
resp = requests.get(f"{BASE_URL}/health")
resp.raise_for_status()
print("API status:", resp.json())

In [ ]:
# Cell 2: Fetch aggregate speed data
params = {"day": "Monday", "period": "AM Peak"}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
features = response.json()
print(f"Received {len(features)} road segments for {params['day']} — {params['period']}")

In [ ]:
# Cell 3: Build GeoDataFrame for inspection
rows = []
for f in features:
    geom = shape(f["geometry"])
    rows.append({
        "link_id": f["link_id"],
        "road_name": f.get("road_name") or "Unknown",
        "length": f.get("length"),
        "average_speed": f["average_speed"],
        "geometry": geom,
    })

gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")
print(f"Speed range: {gdf.average_speed.min():.1f} – {gdf.average_speed.max():.1f} mph")
gdf.describe()

In [ ]:
# Cell 4: Tabular summary — 10 slowest segments
gdf.sort_values("average_speed")[["link_id", "road_name", "average_speed", "length"]].head(10)

In [ ]:
# Cell 5: MapboxGL visualization — speed-colored road segments
# Build GeoJSON feature list for mapboxgl
geojson_features = [
    {
        "type": "Feature",
        "geometry": row.geometry.__geo_interface__,
        "properties": {
            "link_id": row.link_id,
            "road_name": row.road_name,
            "average_speed": round(row.average_speed, 1),
        },
    }
    for _, row in gdf.iterrows()
]

# Color scale: red (slow) → yellow → green (fast)
color_stops = create_color_stops(
    [0, 10, 20, 30, 45, 65],
    colors=["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850", "#006837"],
)

viz = LinestringViz(
    geojson_features,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=color_stops,
    line_width=2.5,
    center=(-81.6557, 30.3322),  # Jacksonville, FL
    zoom=11,
    opacity=0.9,
    legend_layout="horizontal",
    legend_key_shape="line",
    legend_gradient=True,
)
viz.show()

In [ ]:
# Cell 6: Slow links pattern detection
slow_resp = requests.get(
    f"{BASE_URL}/patterns/slow_links/",
    params={"period": "AM Peak", "threshold": 25, "min_days": 1},
)
slow_resp.raise_for_status()
slow_features = slow_resp.json()
print(f"Found {len(slow_features)} segments averaging below 25 mph during AM Peak")

if slow_features:
    slow_geojson = [
        {
            "type": "Feature",
            "geometry": f["geometry"],
            "properties": {
                "link_id": f["link_id"],
                "road_name": f.get("road_name") or "Unknown",
                "average_speed": round(f["average_speed"], 1),
                "days_slow": f["days_slow"],
            },
        }
        for f in slow_features
    ]

    slow_viz = LinestringViz(
        slow_geojson,
        access_token=MAPBOX_TOKEN,
        color_property="average_speed",
        color_stops=create_color_stops([0, 5, 10, 15, 20, 25], colors=["#67001f", "#d6604d", "#f4a582", "#fddbc7", "#f7f7f7", "#d1e5f0"]),
        line_width=3,
        center=(-81.6557, 30.3322),
        zoom=11,
        opacity=1.0,
        legend_layout="horizontal",
        legend_key_shape="line",
        legend_gradient=True,
    )
    slow_viz.show()

In [ ]:
# Cell 7: Spatial filter — downtown Jacksonville bounding box
bbox_resp = requests.post(
    f"{BASE_URL}/aggregates/spatial_filter/",
    json={
        "day": "Monday",
        "period": "AM Peak",
        "bbox": [-81.70, 30.30, -81.60, 30.40],
    },
)
bbox_resp.raise_for_status()
bbox_features = bbox_resp.json()
print(f"Found {len(bbox_features)} segments in downtown Jacksonville bbox")

if bbox_features:
    bbox_geojson = [
        {
            "type": "Feature",
            "geometry": f["geometry"],
            "properties": {
                "link_id": f["link_id"],
                "road_name": f.get("road_name") or "Unknown",
                "average_speed": round(f["average_speed"], 1),
            },
        }
        for f in bbox_features
    ]

    bbox_viz = LinestringViz(
        bbox_geojson,
        access_token=MAPBOX_TOKEN,
        color_property="average_speed",
        color_stops=color_stops,
        line_width=2.5,
        center=(-81.65, 30.35),
        zoom=13,
        opacity=0.9,
        legend_layout="horizontal",
        legend_key_shape="line",
        legend_gradient=True,
    )
    bbox_viz.show()